# Encoder–decoder Transformer with RoPE for legal continuation

## 1. Hugging Face Login

In [1]:
%pip install -q huggingface_hub ijson

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## 2. Check GPU

In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## 4. Load Tokenizer

In [ ]:
from collections import defaultdict, Counter
import heapq

SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class BytelevelBPE:

    def __init__(self, vocab_size=5000):
        self.vocab_size = vocab_size
        self.specials = SPECIALS.copy()
        self.byte_vocab = [str(i) for i in range(256)]
        self.merges = []
        self.merge_ranks = {}
        self.vocab = []
        self.stoi = {}
        self.itos = {}
        # Maps each BPE symbol to the bytes represented by it
        self.symbol_to_bytes = {}
    
    # 1. WORD -> UTF-8 BYTE SYMBOLS

    def word_to_symbols(self, word):
        byte_seq = word.encode("utf-8")
        return [str(b) for b in byte_seq] + ["</w>"]

    # 2. BUILD WORD FREQUENCIES

    def build_word_freqs(self, corpus):
        word_counter = Counter()
        print("Building word frequencies...")
        for text in corpus:
            if not isinstance(text, str):
                continue

            word_counter.update(text.split())
        print(
            f"Unique whitespace-separated words: "
            f"{len(word_counter):,}"
        )
        # Convert each unique word to byte symbols only once
        word_freqs = {}

        for word, freq in word_counter.items():
            symbols = tuple(self.word_to_symbols(word))
            word_freqs[symbols] = (
                word_freqs.get(symbols, 0) + freq
            )

        return word_freqs


    # 3. PAIRS OCCURRING IN ONE WORD

    @staticmethod
    def get_pairs(symbols):
        return {
            (symbols[i], symbols[i + 1])
            for i in range(len(symbols) - 1)
        }

    # 4. COUNT PAIR OCCURRENCES IN ONE WORD

    @staticmethod
    def pair_counts_in_word(symbols):
        counts = Counter()
        for i in range(len(symbols) - 1):
            counts[
                (symbols[i], symbols[i + 1])
            ] += 1

        return counts

    # 5. MERGE A PAIR INSIDE ONE WORD

    @staticmethod
    def merge_symbols(symbols, pair, merged_symbol):
        new_symbols = []
        i = 0
        while i < len(symbols):
            if (
                i < len(symbols) - 1
                and symbols[i] == pair[0]
                and symbols[i + 1] == pair[1]
            ):
                new_symbols.append(merged_symbol)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1

        return tuple(new_symbols)

    # 6. TRAIN OPTIMIZED BPE

    def train(self, corpus):
        word_freqs = self.build_word_freqs(corpus)
        vocab = (
            self.specials
            + self.byte_vocab
            + ["</w>"]
        )
        seen = set(vocab)
        for b in range(256):
            self.symbol_to_bytes[str(b)] = bytes([b])
        self.symbol_to_bytes["</w>"] = b""

        words = []
        frequencies = []
        for symbols, freq in word_freqs.items():
            words.append(symbols)
            frequencies.append(freq)
        del word_freqs

        pair_counts = defaultdict(int)
        pair_to_words = defaultdict(set)

        print("Building initial pair statistics...")

        for word_id, symbols in enumerate(words):
            freq = frequencies[word_id]
            local_counts = self.pair_counts_in_word(symbols)
            for pair, count in local_counts.items():
                pair_counts[pair] += count * freq
                pair_to_words[pair].add(word_id)

        print(
            f"Initial unique pairs: "
            f"{len(pair_counts):,}"
        )

        # MAX HEAP
        #
        # Python has min-heap, so use negative frequency.
        #
        # We keep stale entries and validate them when popped.
        # This avoids rebuilding the heap after every merge.

        heap = []
        for pair, count in pair_counts.items():
            heapq.heappush(heap,(-count, pair))
        target_merges = max(0,self.vocab_size - len(vocab))
        print(f"Target merges: {target_merges:,}")
        while (len(vocab) < self.vocab_size and heap):
            while heap:
                neg_count, pair = heapq.heappop(heap)
                heap_count = -neg_count
                current_count = pair_counts.get(pair,0)
                # Ignore stale heap entries
                if (current_count == heap_count and current_count > 0):
                    break
            else:
                break

            max_freq = pair_counts.get(pair, 0)

            # Same stopping condition as before
            if max_freq <= 1:
                break

            # Create collision-free merged symbol
            merged_symbol = (f"<bpe_{len(self.merges)}>")

            # Byte representation of new symbol
            left_bytes = self.symbol_to_bytes[pair[0]]
            right_bytes = self.symbol_to_bytes[pair[1]]
            self.symbol_to_bytes[merged_symbol] = left_bytes + right_bytes

            self.merges.append(pair)
            self.merge_ranks[pair] = (len(self.merges) - 1,merged_symbol)
            if merged_symbol not in seen:
                seen.add(merged_symbol)
                vocab.append(merged_symbol)

            # ----------------------------------------------------
            # IMPORTANT OPTIMIZATION
            #
            # Only process words containing selected pair.
            # ----------------------------------------------------

            affected_word_ids = list(pair_to_words.get(pair, set()))
            changed_pairs = set()
            for word_id in affected_word_ids:
                old_symbols = words[word_id]
                freq = frequencies[word_id]
                # Pair may have disappeared because of an earlier
                # update involving this word
                if pair not in self.get_pairs(old_symbols):
                    continue
                
                # Statistics BEFORE merge
                old_counts = self.pair_counts_in_word(old_symbols)

                # Merge selected pair
                new_symbols = self.merge_symbols(old_symbols,pair,merged_symbol)
                if new_symbols == old_symbols:
                    continue

                # Statistics AFTER merge
                new_counts = self.pair_counts_in_word(new_symbols)

                # Save updated word
                words[word_id] = new_symbols

                all_pairs = (
                    set(old_counts)
                    | set(new_counts)
                )

                # Update only pairs affected by this word

                for p in all_pairs:

                    old_n = old_counts.get(p, 0)
                    new_n = new_counts.get(p, 0)
                    delta = (
                        new_n - old_n
                    ) * freq

                    if delta != 0:
                        pair_counts[p] += delta
                        changed_pairs.add(p)

                    # Update inverted index
                    if new_n > 0:
                        pair_to_words[p].add(word_id)
                    else:
                        pair_to_words[p].discard(word_id)

            # Selected pair should now be gone from affected words
            pair_to_words[pair].clear()

            # Push updated frequencies into heap
            for p in changed_pairs:
                count = pair_counts.get(p, 0)
                if count > 0:
                    heapq.heappush(heap,(-count, p))

        self.vocab = vocab
        self.vocab_size = len(vocab)
        self.stoi = {
            symbol: i
            for i, symbol in enumerate(vocab)
        }

        self.itos = {
            i: symbol
            for symbol, i in self.stoi.items()
        }

        print("\nTokenizer training complete.")

        print(
            "Final vocabulary size:",
            self.vocab_size
        )

        print(
            "Number of merges:",
            len(self.merges)
        )

    # 7. APPLY BPE TO ONE WORD

    def apply_bpe(self, word):
        symbols = self.word_to_symbols(word)
        while len(symbols) > 1:
            best_pair = None
            best_rank = float("inf")
            best_merged_symbol = None

            for i in range(len(symbols) - 1):
                pair = (symbols[i],symbols[i + 1])
                merge_info = self.merge_ranks.get(pair)
                if merge_info is not None:
                    rank, merged_symbol = merge_info
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = pair
                        best_merged_symbol = merged_symbol

            if best_pair is None:
                break

            symbols = list(
                self.merge_symbols(
                    tuple(symbols),
                    best_pair,
                    best_merged_symbol
                )
            )

        return symbols

    # 8. ENCODE

    def encode(self,text,add_bos=False,add_eos=False):
        ids = []
        if add_bos:
            ids.append(
                self.stoi["<s>"]
            )

        for word in text.split():
            symbols = self.apply_bpe(word)
            for symbol in symbols:
                ids.append(
                    self.stoi.get(
                        symbol,
                        self.stoi["<unk>"]
                    )
                )

        if add_eos:
            ids.append(
                self.stoi["</s>"]
            )

        return ids

    # 9. DECODE

    def decode(self, ids):

        words = []
        current_bytes = bytearray()

        for token_id in ids:
            symbol = self.itos.get(token_id)
            if symbol is None:
                continue
            if symbol in self.specials:
                continue

            # Determine whether this token ends a word
            ends_word = self._symbol_ends_word(symbol)
            byte_value = self.symbol_to_bytes.get(symbol, b"")
            current_bytes.extend(byte_value)

            if ends_word:
                try:
                    word = bytes(current_bytes).decode("utf-8")
                    words.append(word)
                except UnicodeDecodeError:
                    words.append("<unk>")
                current_bytes = bytearray()

        # Remaining bytes
        if current_bytes:
            try:
                words.append(bytes(current_bytes).decode("utf-8"))
            except UnicodeDecodeError:
                words.append("<unk>")

        return " ".join(words)

    # 10. CHECK WHETHER SYMBOL CONTAINS </w> AT END

    def _symbol_ends_word(self, symbol):
        if symbol == "</w>":
            return True
        # Find merge that created this symbol
        for pair, (_, merged) in self.merge_ranks.items():
            if merged == symbol:
                return self._symbol_ends_word(
                    pair[1]
                )

        return False

    # 11. VOCAB SIZE

    def get_vocab_size(self):
        return len(self.vocab)

## 9. Define The Model

In [5]:
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class TransformerConfig:
    vocab_size: int
    pad_id: int
    d_model: int = 512
    n_heads: int = 8
    d_ff: int = 2048
    n_layers: int = 6
    dropout: float = 0.1
    max_src_len: int = 2048
    max_tgt_len: int = 2048
    pos_emb_type: str = "rope"
    rope_base: float = 10000.0


class EmbeddingDropout(nn.Module):
    # Preserve the embedding dropout and configured length limit without adding PE.
    def __init__(self, max_len, dropout):
        super().__init__()
        self.max_len = max_len
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        if x.size(1) > self.max_len:
            raise ValueError('Sequence exceeds configured context length')
        return self.dropout(x)


class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, base=10000.0):
        super().__init__()
        if head_dim % 2:
            raise ValueError('RoPE requires an even head dimension')
        inv_freq = base ** (
            -torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim
        )
        self.register_buffer('inv_freq', inv_freq, persistent=False)

    def forward(self, x):
        # x: [batch, heads, length, head_dim]. Split-half pairs: (i, i + d/2).
        # Generation recomputes the full decoder sequence, so positions start at 0.
        with torch.autocast(device_type=x.device.type, enabled=False):
            positions = torch.arange(x.size(2), device=x.device, dtype=torch.float32)
            angles = torch.outer(positions, self.inv_freq.float())
            cos = angles.cos()[None, None, :, :]
            sin = angles.sin()[None, None, :, :]
        cos, sin = cos.to(x.dtype), sin.to(x.dtype)
        first, second = x.chunk(2, dim=-1)
        return torch.cat((first * cos - second * sin,
                          first * sin + second * cos), dim=-1)


In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout, use_rope=False, rope_base=10000.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.rope = RotaryEmbedding(self.d_head, rope_base) if use_rope else None
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        # [batch, length, d_model] -> [batch, heads, length, d_head]
        b, length, _ = x.shape
        return x.reshape(b, length, self.n_heads, self.d_head).transpose(1, 2)

    def forward(self, query, key, value, allowed, return_attention=False):
        q = self.split_heads(self.q_proj(query))
        k = self.split_heads(self.k_proj(key))
        v = self.split_heads(self.v_proj(value))
        if self.rope is not None:
            q = self.rope(q)
            k = self.rope(k)
        # True means this key may be attended to, including the causal mask.
        weights = None
        if return_attention:
            # Keep explicit probabilities for attention visualizations.
            with torch.autocast(device_type=q.device.type, enabled=False):
                scores = (q.float() @ k.float().transpose(-2, -1)) / math.sqrt(self.d_head)
                scores = scores.masked_fill(~allowed, float('-inf'))
                weights = scores.softmax(dim=-1)
            context = self.dropout(weights.to(v.dtype)) @ v
        else:
            # Let PyTorch select a supported backend; dropout must be 0 at eval.
            context = F.scaled_dot_product_attention(
                q, k, v, attn_mask=allowed,
                dropout_p=self.dropout.p if self.training else 0.0,
                is_causal=False,  # Causality is already present in allowed.
            )
        b, _, length, _ = context.shape
        context = context.transpose(1, 2).contiguous().reshape(b, length, -1)
        output = self.out_proj(context)
        return output, weights.mean(dim=1) if return_attention else None

In [7]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(F.relu(self.linear1(x)))

class EncoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attention = MultiHeadAttention(
            cfg.d_model, cfg.n_heads, cfg.dropout, use_rope=True, rope_base=cfg.rope_base)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff)
        self.norm1 = nn.LayerNorm(cfg.d_model)
        self.norm2 = nn.LayerNorm(cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, source_mask):
        attended, _ = self.attention(x, x, x, source_mask)
        x = self.norm1(x + self.dropout(attended))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.self_attention = MultiHeadAttention(
            cfg.d_model, cfg.n_heads, cfg.dropout, use_rope=True, rope_base=cfg.rope_base)
        self.cross_attention = MultiHeadAttention(
            cfg.d_model, cfg.n_heads, cfg.dropout, use_rope=False)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff)
        self.norm1 = nn.LayerNorm(cfg.d_model)
        self.norm2 = nn.LayerNorm(cfg.d_model)
        self.norm3 = nn.LayerNorm(cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, memory, target_mask, source_mask, return_attention=False):
        attended, _ = self.self_attention(x, x, x, target_mask)
        x = self.norm1(x + self.dropout(attended))
        attended, weights = self.cross_attention(x, memory, memory, source_mask,
                                                  return_attention)
        x = self.norm2(x + self.dropout(attended))
        x = self.norm3(x + self.dropout(self.ffn(x)))
        return x, weights

In [8]:
class LegalTransformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        if cfg.pos_emb_type != 'rope':
            raise ValueError('This architecture expects a RoPE configuration')
        self.cfg = cfg
        # Separate weights make the three roles easy to inspect.
        self.source_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)
        self.target_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)
        self.source_position = EmbeddingDropout(cfg.max_src_len, cfg.dropout)
        self.target_position = EmbeddingDropout(cfg.max_tgt_len, cfg.dropout)
        self.encoder = nn.ModuleList([EncoderLayer(cfg) for _ in range(cfg.n_layers)])
        self.decoder = nn.ModuleList([DecoderLayer(cfg) for _ in range(cfg.n_layers)])
        self.output = nn.Linear(cfg.d_model, cfg.vocab_size)
        for parameter in self.parameters():
            if parameter.dim() > 1:
                nn.init.xavier_uniform_(parameter)
        with torch.no_grad():
            self.source_embedding.weight[cfg.pad_id].zero_()
            self.target_embedding.weight[cfg.pad_id].zero_()

    def encode(self, source_ids):
        source_mask = source_ids.ne(self.cfg.pad_id)[:, None, None, :]
        x = self.source_embedding(source_ids) * math.sqrt(self.cfg.d_model)
        x = self.source_position(x)
        for layer in self.encoder:
            x = layer(x, source_mask)
        return x, source_mask

    def decode(self, target_ids, memory, source_mask, return_attention=False):
        length = target_ids.size(1)
        causal = torch.ones(length, length, dtype=torch.bool, device=target_ids.device).tril()
        valid_keys = target_ids.ne(self.cfg.pad_id)[:, None, None, :]
        target_mask = valid_keys & causal[None, None, :, :]
        x = self.target_embedding(target_ids) * math.sqrt(self.cfg.d_model)
        x = self.target_position(x)
        weights = None
        for index, layer in enumerate(self.decoder):
            x, weights = layer(x, memory, target_mask, source_mask,
                               return_attention and index == len(self.decoder) - 1)
        logits = self.output(x)
        return (logits, weights) if return_attention else logits

    def forward(self, source_ids, target_ids, return_attention=False):
        memory, source_mask = self.encode(source_ids)
        return self.decode(target_ids, memory, source_mask, return_attention)

# **Phase 5**

In [9]:
import os
import json
import random
import zipfile
from pathlib import Path
from dataclasses import asdict
import numpy as np
import torch
import torch.nn as nn
from huggingface_hub import hf_hub_download
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint as activation_checkpoint
from torch.nn.utils.rnn import (
    pad_sequence,
    pack_padded_sequence,
    pad_packed_sequence,
)
from sklearn.metrics import f1_score, accuracy_score

MODEL_DIR = Path(
    "/kaggle/input/datasets/tejasvaidya137/model2-for-phase5-files"
)

OUTPUT_DIR = Path("/kaggle/working/phase5_encdec_lora")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42

CHUNK_LENGTH = 512
MAX_CHUNKS = 16
DOC_BATCH_SIZE = 8
CHUNK_BATCH_SIZE = 32

ENCODER_ADAPT_LAST_N = 0
DECODER_ADAPT_LAST_N = 2

RANK = 16
ALPHA = 32

ATTENTION_TARGETS = [
    "q_proj", "k_proj", "v_proj", "out_proj"
]
FFN_TARGETS = ["linear1", "linear2"]

GRU_HIDDEN = 256
HEAD_DROPOUT = 0.2

LORA_LR = 1e-4
HEAD_LR = 1e-4

EPOCHS = 5
PATIENCE = 3
USE_ACTIVATION_CHECKPOINTING = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

use_amp = device.type == "cuda"

amp_dtype = (
    torch.bfloat16
    if use_amp and torch.cuda.is_bf16_supported()
    else torch.float16
)

In [10]:
saved = torch.load(
    MODEL_DIR / "model.pt",
    map_location="cpu",
    weights_only=True,
)

cfg = TransformerConfig(**saved["model_config"])

phase1_model = LegalTransformer(cfg)
phase1_model.load_state_dict(saved["model_state_dict"])

del saved

with open(MODEL_DIR / "tokenizer.json", encoding="utf-8") as f:
    state = json.load(f)

tokenizer = BytelevelBPE(vocab_size=state["vocab_size"])
tokenizer.specials = state["specials"]
tokenizer.vocab = state["vocab"]
tokenizer.vocab_size = len(tokenizer.vocab)

tokenizer.stoi = {
    symbol: i for i, symbol in enumerate(tokenizer.vocab)
}
tokenizer.itos = {
    i: symbol for symbol, i in tokenizer.stoi.items()
}
tokenizer.merges = [
    tuple(pair) for pair in state["merges"]
]
tokenizer.merge_ranks = {
    tuple(record["pair"]): (
        record["rank"],
        record["merged_symbol"],
    )
    for record in state["merge_ranks"]
}
tokenizer.symbol_to_bytes = {
    symbol: bytes(values)
    for symbol, values in state["symbol_to_bytes"].items()
}

del state

PAD_ID = tokenizer.stoi["<pad>"]
UNK_ID = tokenizer.stoi["<unk>"]
BOS_ID = tokenizer.stoi["<s>"]
EOS_ID = tokenizer.stoi["</s>"]

CHUNK_LENGTH = min(
    CHUNK_LENGTH,
    cfg.max_src_len,
    cfg.max_tgt_len,
)

In [11]:
DATA_DIR = Path("/kaggle/working/capstone_data")

for filename in [
    "Phase_5_LSI/Train/train.json",
    "Phase_5_LSI/Train/dev.json",
    "Phase_5_LSI/Test/test.json",
]:
    hf_hub_download(
        repo_id="Exploration-Lab/CS-781-Capstone",
        repo_type="dataset",
        filename=filename,
        local_dir=str(DATA_DIR),
        token=globals().get("HF_TOKEN") or os.getenv("HF_TOKEN"),
    )

LSI_DIR = DATA_DIR / "Phase_5_LSI"


def read_cases(path, labeled=True):
    with open(path, encoding="utf-8") as f:
        records = json.load(f)

    cases = []
    seen = {}

    for record in records:
        labels = (
            sorted({int(label) for label in record["labels"]})
            if labeled else []
        )

        case = {
            "id": record["id"],
            "text": record["text"],
            "labels": labels,
        }

        if labeled:
            key = str(case["id"])
            value = (case["text"], tuple(labels))

            if key in seen:
                if seen[key] != value:
                    raise ValueError(f"Conflicting case ID: {key}")
                continue

            seen[key] = value

        cases.append(case)

    return cases


train_cases = read_cases(LSI_DIR / "Train/train.json")
dev_cases = read_cases(LSI_DIR / "Train/dev.json")

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

In [12]:
statute_ids = sorted({
    label
    for case in train_cases + dev_cases
    for label in case["labels"]
})

label2index = {
    label: index for index, label in enumerate(statute_ids)
}

NUM_LABELS = len(statute_ids)


def make_target(labels):
    target = torch.zeros(NUM_LABELS, dtype=torch.float32)

    for label in labels:
        target[label2index[label]] = 1.0

    return target

In [13]:
class StatuteDataset(Dataset):
    def __init__(self, cases):
        self.cases = cases
        self.chunks = []
        self.targets = [
            make_target(case["labels"]) for case in cases
        ]

        capacity = CHUNK_LENGTH - 2

        for i, case in enumerate(cases):
            ids = tokenizer.encode(case["text"]) or [UNK_ID]
            starts = list(range(0, len(ids), capacity))

            if len(starts) > MAX_CHUNKS:
                selected = np.linspace(
                    0, len(starts) - 1, MAX_CHUNKS, dtype=int
                )
                starts = [starts[j] for j in selected]

            self.chunks.append([
                np.array(
                    [BOS_ID]
                    + ids[start:start + capacity]
                    + [EOS_ID],
                    dtype=np.int32,
                )
                for start in starts
            ])

            if (i + 1) % 1000 == 0:
                print(f"Tokenized {i + 1}/{len(cases)}")

    def __len__(self):
        return len(self.cases)

    def __getitem__(self, index):
        return {
            "id": self.cases[index]["id"],
            "labels": self.targets[index],
            "chunks": [
                torch.tensor(chunk, dtype=torch.long)
                for chunk in self.chunks[index]
            ],
        }


def collate_cases(examples):
    chunks = [
        chunk
        for example in examples
        for chunk in example["chunks"]
    ]

    return {
        "input_ids": pad_sequence(
            chunks,
            batch_first=True,
            padding_value=PAD_ID,
        ),
        "lengths": torch.tensor([
            len(chunk) for chunk in chunks
        ]),
        "chunk_counts": torch.tensor([
            len(example["chunks"]) for example in examples
        ]),
        "labels": torch.stack([
            example["labels"] for example in examples
        ]),
        "ids": [example["id"] for example in examples],
    }


def make_loader(dataset, shuffle=False):
    return DataLoader(
        dataset,
        batch_size=DOC_BATCH_SIZE,
        shuffle=shuffle,
        collate_fn=collate_cases,
        pin_memory=use_amp,
    )


def move_batch(batch):
    for key in ["input_ids", "lengths", "labels"]:
        batch[key] = batch[key].to(
            device, non_blocking=True
        )

    # Keep chunk_counts on CPU for pack_padded_sequence.
    return batch


train_dataset = StatuteDataset(train_cases)
dev_dataset = StatuteDataset(dev_cases)

train_loader = make_loader(train_dataset, shuffle=True)
dev_loader = make_loader(dev_dataset)

Tokenized 1000/42750
Tokenized 2000/42750
Tokenized 3000/42750
Tokenized 4000/42750
Tokenized 5000/42750
Tokenized 6000/42750
Tokenized 7000/42750
Tokenized 8000/42750
Tokenized 9000/42750
Tokenized 10000/42750
Tokenized 11000/42750
Tokenized 12000/42750
Tokenized 13000/42750
Tokenized 14000/42750
Tokenized 15000/42750
Tokenized 16000/42750
Tokenized 17000/42750
Tokenized 18000/42750
Tokenized 19000/42750
Tokenized 20000/42750
Tokenized 21000/42750
Tokenized 22000/42750
Tokenized 23000/42750
Tokenized 24000/42750
Tokenized 25000/42750
Tokenized 26000/42750
Tokenized 27000/42750
Tokenized 28000/42750
Tokenized 29000/42750
Tokenized 30000/42750
Tokenized 31000/42750
Tokenized 32000/42750
Tokenized 33000/42750
Tokenized 34000/42750
Tokenized 35000/42750
Tokenized 36000/42750
Tokenized 37000/42750
Tokenized 38000/42750
Tokenized 39000/42750
Tokenized 40000/42750
Tokenized 41000/42750
Tokenized 42000/42750
Tokenized 1000/10181
Tokenized 2000/10181
Tokenized 3000/10181
Tokenized 4000/10181
T

In [14]:
class LoRA(nn.Module):
    def __init__(self, base, rank, alpha):
        super().__init__()

        self.base = base
        self.base.requires_grad_(False)
        self.scale = alpha / rank

        self.A = nn.Linear(
            base.in_features,
            rank,
            bias=False,
            device=base.weight.device,
            dtype=base.weight.dtype,
        )

        self.B = nn.Linear(
            rank,
            base.out_features,
            bias=False,
            device=base.weight.device,
            dtype=base.weight.dtype,
        )

        nn.init.normal_(self.A.weight, std=0.02)
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        return self.base(x) + self.scale * self.B(self.A(x))


def add_lora(module, names):
    for name in names:
        layer = getattr(module, name)
        setattr(module, name, LoRA(layer, RANK, ALPHA))


class EncoderDecoderLSI(nn.Module):
    def __init__(self, base, num_labels):
        super().__init__()

        base.requires_grad_(False)

        self.source_embedding = base.source_embedding
        self.target_embedding = base.target_embedding
        self.source_position = base.source_position
        self.target_position = base.target_position

        self.encoder = base.encoder
        self.decoder = base.decoder

        self.embedding_scale = base.cfg.d_model ** 0.5
        self.chunk_batch_size = CHUNK_BATCH_SIZE
        self.use_checkpointing = USE_ACTIVATION_CHECKPOINTING

        self.encoder_start = max(
            0, len(self.encoder) - ENCODER_ADAPT_LAST_N
        )
        self.decoder_start = max(
            0, len(self.decoder) - DECODER_ADAPT_LAST_N
        )

        for layer in self.encoder[self.encoder_start:]:
            add_lora(layer.attention, ATTENTION_TARGETS)
            add_lora(layer.ffn, FFN_TARGETS)

        for layer in self.decoder[self.decoder_start:]:
            add_lora(layer.self_attention, ATTENTION_TARGETS)
            add_lora(layer.cross_attention, ATTENTION_TARGETS)
            add_lora(layer.ffn, FFN_TARGETS)

        self.gru = nn.GRU(
            input_size=base.cfg.d_model,
            hidden_size=GRU_HIDDEN,
            batch_first=True,
            bidirectional=True,
        )

        self.attention = nn.Sequential(
            nn.Linear(2 * GRU_HIDDEN, 128),
            nn.Tanh(),
            nn.Linear(128, 1, bias=False),
        )

        self.dropout = nn.Dropout(HEAD_DROPOUT)
        self.classifier = nn.Linear(
            2 * GRU_HIDDEN, num_labels
        )

    def train(self, mode=True):
        super().train(mode)

        self.source_position.eval()
        self.target_position.eval()

        for layer in self.encoder[:self.encoder_start]:
            layer.eval()

        for layer in self.decoder[:self.decoder_start]:
            layer.eval()

        return self

    def encode_chunks(self, input_ids, lengths):
        length = input_ids.size(1)

        positions = torch.arange(
            length, device=input_ids.device
        )

        valid = positions[None, :] < lengths[:, None]
        source_mask = valid[:, None, None, :]

        memory = self.source_position(
            self.source_embedding(input_ids)
            * self.embedding_scale
        )

        for i, layer in enumerate(self.encoder):
            if (
                self.training
                and self.use_checkpointing
                and i >= self.encoder_start
            ):
                memory = activation_checkpoint(
                    layer,
                    memory,
                    source_mask,
                    use_reentrant=False,
                )
            else:
                memory = layer(memory, source_mask)

        causal = (
            positions[None, :] <= positions[:, None]
        )
        target_mask = (
            source_mask & causal[None, None, :, :]
        )

        x = self.target_position(
            self.target_embedding(input_ids)
            * self.embedding_scale
        )

        for i, layer in enumerate(self.decoder):
            if (
                self.training
                and self.use_checkpointing
                and (
                    i >= self.decoder_start
                    or memory.requires_grad
                )
            ):
                x, _ = activation_checkpoint(
                    layer,
                    x,
                    memory,
                    target_mask,
                    source_mask,
                    use_reentrant=False,
                )
            else:
                x, _ = layer(
                    x,
                    memory,
                    target_mask,
                    source_mask,
                )

        x = x.float()

        # Exclude BOS, EOS, and padding.
        text_mask = (
            (positions[None, :] >= 1)
            & (positions[None, :] < lengths[:, None] - 1)
        ).unsqueeze(-1).float()

        return (
            (x * text_mask).sum(1)
            / text_mask.sum(1).clamp_min(1)
        )

    def forward(self, input_ids, lengths, chunk_counts):
        vectors = []

        for start in range(
            0, len(input_ids), self.chunk_batch_size
        ):
            end = start + self.chunk_batch_size

            vectors.append(
                self.encode_chunks(
                    input_ids[start:end],
                    lengths[start:end],
                )
            )

        vectors = torch.cat(vectors)

        sequences = torch.split(
            vectors, chunk_counts.tolist()
        )
        x = pad_sequence(sequences, batch_first=True)

        with torch.autocast(
            device_type=x.device.type,
            enabled=False,
        ):
            x = self.dropout(x.float())

            packed = pack_padded_sequence(
                x,
                chunk_counts,
                batch_first=True,
                enforce_sorted=False,
            )

            packed, _ = self.gru(packed)

            x, _ = pad_packed_sequence(
                packed, batch_first=True
            )

            positions = torch.arange(
                x.size(1), device=x.device
            )[None, :]

            valid = (
                positions
                < chunk_counts.to(x.device)[:, None]
            )

            scores = self.attention(x).squeeze(-1)
            scores = scores.masked_fill(
                ~valid, float("-inf")
            )

            weights = torch.softmax(scores, dim=1)

            document_vectors = (
                x * weights.unsqueeze(-1)
            ).sum(1)

            return self.classifier(
                self.dropout(document_vectors)
            )


torch.manual_seed(SEED)

lsi_model = EncoderDecoderLSI(
    phase1_model, NUM_LABELS
).to(device)

del phase1_model

In [15]:
lora_parameters = []
head_parameters = []

for name, parameter in lsi_model.named_parameters():
    if not parameter.requires_grad:
        continue

    if name.startswith(("encoder.", "decoder.")):
        lora_parameters.append(parameter)
    else:
        head_parameters.append(parameter)

optimizer = torch.optim.AdamW(
    [
        {
            "params": lora_parameters,
            "lr": LORA_LR,
        },
        {
            "params": head_parameters,
            "lr": HEAD_LR,
        },
    ],
    weight_decay=0.01,
)

criterion = nn.BCEWithLogitsLoss()

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp and amp_dtype == torch.float16,
)

In [16]:
def get_logits(model, batch):
    return model(
        batch["input_ids"],
        batch["lengths"],
        batch["chunk_counts"],
    ).float()


@torch.inference_mode()
def collect_probabilities(model, loader):
    model.eval()

    probabilities = []
    targets = []

    for batch in loader:
        batch = move_batch(batch)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            logits = get_logits(model, batch)

        probabilities.append(
            torch.sigmoid(logits).cpu()
        )
        targets.append(batch["labels"].cpu())

    return (
        torch.cat(probabilities).numpy(),
        torch.cat(targets).numpy().astype(np.int32),
    )


def choose_threshold(probabilities, targets):
    results = []

    for threshold in np.round(
        np.arange(0.01, 1.00, 0.01), 2
    ):
        predictions = probabilities >= threshold

        score = f1_score(
            targets,
            predictions,
            average="macro",
            zero_division=0,
        )

        results.append(
            (float(threshold), float(score))
        )

    return min(
        results,
        key=lambda row: (-row[1], abs(row[0] - 0.5)),
    )

In [17]:
model_settings = {
    "phase1_config": asdict(cfg),
    "adaptation": "lora",
    "encoder_adapt_last_n": ENCODER_ADAPT_LAST_N,
    "decoder_adapt_last_n": DECODER_ADAPT_LAST_N,
    "rank": RANK,
    "alpha": ALPHA,
    "attention_targets": ATTENTION_TARGETS.copy(),
    "ffn_targets": FFN_TARGETS.copy(),
    "decoder_causal": True,
    "gru_hidden": GRU_HIDDEN,
    "head_dropout": HEAD_DROPOUT,
    "chunk_length": CHUNK_LENGTH,
    "max_chunks": MAX_CHUNKS,
}

best_f1 = -1.0
bad_epochs = 0

for epoch in range(1, EPOCHS + 1):
    lsi_model.train()

    total_loss = 0.0
    total_cases = 0

    for step, batch in enumerate(train_loader, start=1):
        batch = move_batch(batch)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            logits = get_logits(lsi_model, batch)
            loss = criterion(logits, batch["labels"])

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        nn.utils.clip_grad_norm_(
            lsi_model.parameters(), 1.0
        )

        scaler.step(optimizer)
        scaler.update()

        batch_size = len(batch["ids"])
        total_loss += loss.item() * batch_size
        total_cases += batch_size

        if step % 100 == 0:
            print(
                f"Epoch {epoch} | "
                f"Batch {step}/{len(train_loader)} | "
                f"Loss: {total_loss / total_cases:.4f}"
            )

    dev_probs, dev_targets = collect_probabilities(
        lsi_model, dev_loader
    )

    threshold, dev_f1 = choose_threshold(
        dev_probs, dev_targets
    )

    print(
        f"Epoch {epoch} | "
        f"Train loss: {total_loss / total_cases:.4f} | "
        f"Dev macro F1: {dev_f1:.4f} | "
        f"Threshold: {threshold:.2f}"
    )

    improved = dev_f1 > best_f1

    if improved:
        best_f1 = dev_f1
        bad_epochs = 0
    else:
        bad_epochs += 1

    state = {
        "model_state_dict": lsi_model.state_dict(),
        "model_settings": model_settings,
        "statute_ids": statute_ids,
        "threshold": threshold,
        "epoch": epoch,
        "dev_f1": dev_f1,
        "best_f1": best_f1,
        "bad_epochs": bad_epochs,
    }

    if improved:
        torch.save(state, OUTPUT_DIR / "best_f1.pt")
        print("Saved best model.")

    state.update({
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
    })

    torch.save(state, OUTPUT_DIR / "latest.pt")

    if bad_epochs >= PATIENCE:
        print("Early stopping.")
        break

Epoch 1 | Batch 100/5344 | Loss: 0.2742
Epoch 1 | Batch 200/5344 | Loss: 0.2087
Epoch 1 | Batch 300/5344 | Loss: 0.1859
Epoch 1 | Batch 400/5344 | Loss: 0.1743
Epoch 1 | Batch 500/5344 | Loss: 0.1672
Epoch 1 | Batch 600/5344 | Loss: 0.1629
Epoch 1 | Batch 700/5344 | Loss: 0.1587
Epoch 1 | Batch 800/5344 | Loss: 0.1559
Epoch 1 | Batch 900/5344 | Loss: 0.1537
Epoch 1 | Batch 1000/5344 | Loss: 0.1520
Epoch 1 | Batch 1100/5344 | Loss: 0.1502
Epoch 1 | Batch 1200/5344 | Loss: 0.1491
Epoch 1 | Batch 1300/5344 | Loss: 0.1477
Epoch 1 | Batch 1400/5344 | Loss: 0.1459
Epoch 1 | Batch 1500/5344 | Loss: 0.1446
Epoch 1 | Batch 1600/5344 | Loss: 0.1437
Epoch 1 | Batch 1700/5344 | Loss: 0.1429
Epoch 1 | Batch 1800/5344 | Loss: 0.1419
Epoch 1 | Batch 1900/5344 | Loss: 0.1410
Epoch 1 | Batch 2000/5344 | Loss: 0.1401
Epoch 1 | Batch 2100/5344 | Loss: 0.1393
Epoch 1 | Batch 2200/5344 | Loss: 0.1385
Epoch 1 | Batch 2300/5344 | Loss: 0.1377
Epoch 1 | Batch 2400/5344 | Loss: 0.1369
Epoch 1 | Batch 2500/5344

In [18]:
saved = torch.load(
    OUTPUT_DIR / "best_f1.pt",
    map_location="cpu",
    weights_only=True,
)

lsi_model.load_state_dict(saved["model_state_dict"])

BEST_THRESHOLD = float(saved["threshold"])
statute_ids = saved["statute_ids"]

del saved

dev_probs, dev_targets = collect_probabilities(
    lsi_model, dev_loader
)

dev_predictions = (
    dev_probs >= BEST_THRESHOLD
).astype(np.int32)

print("Best threshold:", BEST_THRESHOLD)

print(
    "Macro F1:",
    f1_score(
        dev_targets,
        dev_predictions,
        average="macro",
        zero_division=0,
    ),
)

print(
    "Micro F1:",
    f1_score(
        dev_targets,
        dev_predictions,
        average="micro",
        zero_division=0,
    ),
)

print(
    "Subset accuracy:",
    accuracy_score(dev_targets, dev_predictions),
)

Best threshold: 0.13
Macro F1: 0.2923928874466883
Micro F1: 0.4426046898010044
Subset accuracy: 0.012081327963854239


In [19]:
test_cases = read_cases(
    LSI_DIR / "Test/test.json",
    labeled=False,
)

test_dataset = StatuteDataset(test_cases)
test_loader = make_loader(test_dataset)


@torch.inference_mode()
def predict_statutes(model, loader, threshold):
    model.eval()
    results = []

    for batch in loader:
        batch = move_batch(batch)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            logits = get_logits(model, batch)

        predictions = (
            torch.sigmoid(logits) >= threshold
        ).cpu().numpy()

        for case_id, row in zip(
            batch["ids"], predictions
        ):
            labels = [
                int(statute_ids[index])
                for index in np.flatnonzero(row)
            ]

            results.append({
                "id": case_id,
                "pred": labels,
            })

    return results


test_predictions = predict_statutes(
    lsi_model,
    test_loader,
    BEST_THRESHOLD,
)

prediction_path = OUTPUT_DIR / "prediction.json"

with open(prediction_path, "w", encoding="utf-8") as f:
    json.dump(
        test_predictions,
        f,
        ensure_ascii=False,
        indent=2,
    )

with zipfile.ZipFile(
    OUTPUT_DIR / "submission.zip",
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(
        prediction_path,
        arcname="prediction.json",
    )

print("Saved:", OUTPUT_DIR / "submission.zip")

Tokenized 1000/13019
Tokenized 2000/13019
Tokenized 3000/13019
Tokenized 4000/13019
Tokenized 5000/13019
Tokenized 6000/13019
Tokenized 7000/13019
Tokenized 8000/13019
Tokenized 9000/13019
Tokenized 10000/13019
Tokenized 11000/13019
Tokenized 12000/13019
Tokenized 13000/13019
Saved: /kaggle/working/phase5_encdec_lora/submission.zip
